# Notebook 34 — Prepare Final Dashboard Assets

This notebook prepares dashboard-ready assets for the final pre-feedback restoration evaluation framework.

It combines existing outputs from previous notebooks:

- final refined controlled evaluation,
- texture and brushstroke-proxy diagnostics,
- Stable Diffusion uncertainty heatmaps,
- selected per-case diagnostic reports.

The notebook does not rerun models and does not recompute core metrics.

The output files are designed for `streamlit_app.py`, so the app can load clean CSV/JSON assets instead of reconstructing the project state directly from many notebook outputs.

In [1]:
from pathlib import Path, PureWindowsPath
import json
import os
from datetime import datetime

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 160)
pd.set_option("display.width", 240)

print("Notebook 34 imports ready.")

Notebook 34 imports ready.


In [2]:
NOTEBOOK_NAME = "34_prepare_final_dashboard_assets_cleaned"

REPO_FOLDER_NAME_CANDIDATES = [
    "painting-restoration-eval",
    "painting_restoration_eval",
]


def find_project_root(
    start_path: Path | None = None,
    repo_folder_name_candidates: list[str] | None = None,
) -> Path:
    if repo_folder_name_candidates is None:
        repo_folder_name_candidates = REPO_FOLDER_NAME_CANDIDATES

    if start_path is None:
        start_path = Path.cwd()

    start_path = start_path.resolve()

    for parent in [start_path] + list(start_path.parents):
        if parent.name in repo_folder_name_candidates:
            return parent

        if (
            (parent / ".git").exists()
            and (parent / "notebooks").exists()
            and (parent / "data").exists()
            and (parent / "outputs").exists()
        ):
            return parent

        if (
            (parent / "notebooks").exists()
            and (parent / "data").exists()
            and (parent / "outputs").exists()
        ):
            return parent

    raise RuntimeError(
        f"Could not find project root from {start_path}. "
        f"Tried folder names: {repo_folder_name_candidates}"
    )


PROJECT_ROOT = find_project_root()
REPO_FOLDER_NAME = PROJECT_ROOT.name

outputs_dir = PROJECT_ROOT / "outputs"
metrics_dir = outputs_dir / "metrics"
reports_dir = outputs_dir / "reports"
figures_dir = outputs_dir / "figures"

dashboard_dir = outputs_dir / "dashboard"
dashboard_figures_dir = dashboard_dir / "figures"

dashboard_dir.mkdir(parents=True, exist_ok=True)
dashboard_figures_dir.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Dashboard directory:", dashboard_dir)

Project root: D:\Masters\FH\Thesis\painting-restoration-eval
Dashboard directory: D:\Masters\FH\Thesis\painting-restoration-eval\outputs\dashboard


In [3]:
input_paths = {
    # Core final evaluation.
    "refined_comparison": metrics_dir / "comparison_unified_refined_opencv_lama_stable_diffusion_50.csv",

    # Texture / brushstroke-proxy outputs from Notebook 31.
    "texture_unified": metrics_dir / "comparison_texture_unified_50.csv",
    "texture_case_winners_nonzero": metrics_dir / "comparison_texture_case_winners_nonzero_50.csv",
    "texture_winner_summary_nonzero": metrics_dir / "comparison_texture_winner_summary_nonzero_50.csv",
    "texture_disagreement_cases": metrics_dir / "comparison_texture_disagreement_cases_50.csv",
    "texture_high_texture_brushwork_summary": metrics_dir / "comparison_texture_high_texture_brushwork_summary_50.csv",
    "brushstroke_proxy_summary_by_model": metrics_dir / "comparison_brushstroke_proxy_summary_by_model_50.csv",

    # Stable Diffusion uncertainty heatmap outputs from Notebook 32.
    "uncertainty_heatmap_manifest": metrics_dir / "stable_diffusion_uncertainty_heatmap_manifest_50.csv",
    "uncertainty_heatmap_summary_by_case": metrics_dir / "stable_diffusion_uncertainty_heatmap_summary_by_case_50.csv",
    "uncertainty_heatmap_summary_by_mask_type": metrics_dir / "stable_diffusion_uncertainty_heatmap_summary_by_mask_type_50.csv",
    "uncertainty_heatmap_summary_by_category": metrics_dir / "stable_diffusion_uncertainty_heatmap_summary_by_category_50.csv",
    "uncertainty_heatmap_vs_refined_performance": metrics_dir / "stable_diffusion_uncertainty_heatmap_vs_refined_performance_50.csv",
    "uncertainty_heatmap_selected_cases": metrics_dir / "stable_diffusion_uncertainty_heatmap_selected_cases_50.csv",

    # Case diagnostics from Notebook 33.
    "case_diagnostic_selected_cases": metrics_dir / "case_diagnostic_selected_cases_50.csv",
    "case_diagnostic_report_manifest": metrics_dir / "case_diagnostic_report_manifest_50.csv",

    # Reports.
    "uncertainty_heatmap_report": reports_dir / "stable_diffusion_uncertainty_heatmap_report_50.html",
    "case_report_index": reports_dir / "case_diagnostics" / "case_report_index.html",
}

output_paths = {
    "dashboard_summary": dashboard_dir / "dashboard_summary.json",
    "dashboard_model_winner_summary": dashboard_dir / "dashboard_model_winner_summary.csv",
    "dashboard_metric_vote_summary": dashboard_dir / "dashboard_metric_vote_summary.csv",
    "dashboard_texture_summary": dashboard_dir / "dashboard_texture_summary.csv",
    "dashboard_texture_disagreements": dashboard_dir / "dashboard_texture_disagreements.csv",
    "dashboard_uncertainty_summary": dashboard_dir / "dashboard_uncertainty_summary.csv",
    "dashboard_uncertainty_selected_cases": dashboard_dir / "dashboard_uncertainty_selected_cases.csv",
    "dashboard_case_report_manifest": dashboard_dir / "dashboard_case_report_manifest.csv",
    "dashboard_selected_cases": dashboard_dir / "dashboard_selected_cases.csv",
    "dashboard_figure_manifest": dashboard_dir / "dashboard_figure_manifest.csv",
    "dashboard_asset_manifest": dashboard_dir / "dashboard_asset_manifest.json",
}

print("Input paths:")
for name, path in input_paths.items():
    status = "exists" if path.exists() else "missing"
    print(f"- {name}: {path} [{status}]")

print("\nOutput paths:")
for name, path in output_paths.items():
    print(f"- {name}: {path}")

Input paths:
- refined_comparison: D:\Masters\FH\Thesis\painting-restoration-eval\outputs\metrics\comparison_unified_refined_opencv_lama_stable_diffusion_50.csv [exists]
- texture_unified: D:\Masters\FH\Thesis\painting-restoration-eval\outputs\metrics\comparison_texture_unified_50.csv [exists]
- texture_case_winners_nonzero: D:\Masters\FH\Thesis\painting-restoration-eval\outputs\metrics\comparison_texture_case_winners_nonzero_50.csv [exists]
- texture_winner_summary_nonzero: D:\Masters\FH\Thesis\painting-restoration-eval\outputs\metrics\comparison_texture_winner_summary_nonzero_50.csv [exists]
- texture_disagreement_cases: D:\Masters\FH\Thesis\painting-restoration-eval\outputs\metrics\comparison_texture_disagreement_cases_50.csv [exists]
- texture_high_texture_brushwork_summary: D:\Masters\FH\Thesis\painting-restoration-eval\outputs\metrics\comparison_texture_high_texture_brushwork_summary_50.csv [exists]
- brushstroke_proxy_summary_by_model: D:\Masters\FH\Thesis\painting-restoration-e

In [4]:
def get_repo_folder_candidates() -> list[str]:
    candidates = [
        "painting-restoration-eval",
        "painting_restoration_eval",
        REPO_FOLDER_NAME,
    ]

    return list(dict.fromkeys(candidates))


def to_project_relative_path(path_value: str | Path | None) -> str | None:
    if path_value is None:
        return None

    try:
        if pd.isna(path_value):
            return None
    except TypeError:
        pass

    path_text = str(path_value).strip()

    if not path_text:
        return None

    normalized_text = path_text.replace("\\", "/")

    for repo_folder_name in get_repo_folder_candidates():
        marker = f"/{repo_folder_name}/"

        if marker in normalized_text:
            return normalized_text.split(marker, 1)[1]

        if normalized_text.startswith(f"{repo_folder_name}/"):
            return normalized_text.split(f"{repo_folder_name}/", 1)[1]

    path = Path(normalized_text)

    if not path.is_absolute():
        return path.as_posix()

    try:
        return path.relative_to(PROJECT_ROOT).as_posix()
    except ValueError:
        pass

    windows_path = PureWindowsPath(path_text)

    for repo_folder_name in get_repo_folder_candidates():
        if repo_folder_name in windows_path.parts:
            repo_index = windows_path.parts.index(repo_folder_name)
            relative_parts = windows_path.parts[repo_index + 1:]
            return Path(*relative_parts).as_posix()

    return normalized_text


def resolve_project_path(path_value: str | Path | None) -> Path | None:
    if path_value is None:
        return None

    try:
        if pd.isna(path_value):
            return None
    except TypeError:
        pass

    path_text = str(path_value).strip()

    if not path_text:
        return None

    normalized_text = path_text.replace("\\", "/")

    for repo_folder_name in get_repo_folder_candidates():
        marker = f"/{repo_folder_name}/"

        if marker in normalized_text:
            relative_part = normalized_text.split(marker, 1)[1]
            return PROJECT_ROOT / relative_part

        if normalized_text.startswith(f"{repo_folder_name}/"):
            relative_part = normalized_text.split(f"{repo_folder_name}/", 1)[1]
            return PROJECT_ROOT / relative_part

    windows_path = PureWindowsPath(path_text)

    for repo_folder_name in get_repo_folder_candidates():
        if repo_folder_name in windows_path.parts:
            repo_index = windows_path.parts.index(repo_folder_name)
            relative_parts = windows_path.parts[repo_index + 1:]
            return PROJECT_ROOT.joinpath(*relative_parts)

    path = Path(path_text)

    if path.is_absolute():
        if path.exists():
            return path

        for repo_folder_name in get_repo_folder_candidates():
            if repo_folder_name in path.parts:
                repo_index = path.parts.index(repo_folder_name)
                relative_parts = path.parts[repo_index + 1:]
                return PROJECT_ROOT.joinpath(*relative_parts)

        return path

    return PROJECT_ROOT / path


def file_exists_from_value(path_value: str | Path | None) -> bool:
    resolved_path = resolve_project_path(path_value)
    return bool(resolved_path is not None and resolved_path.exists())


def write_json(path: Path, payload: dict) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(
        json.dumps(payload, indent=2, ensure_ascii=False),
        encoding="utf-8",
    )


print("Path and JSON helpers ready.")

Path and JSON helpers ready.


In [5]:
required_input_keys = [
    "refined_comparison",
    "case_diagnostic_selected_cases",
    "case_diagnostic_report_manifest",
]

missing_required_inputs = [
    key
    for key in required_input_keys
    if not input_paths[key].exists()
]

if missing_required_inputs:
    for key in missing_required_inputs:
        print(f"Missing required input: {key} -> {input_paths[key]}")

    raise FileNotFoundError("One or more required Notebook 34 inputs are missing.")

optional_input_keys = [
    key
    for key in input_paths
    if key not in required_input_keys
]

missing_optional_inputs = [
    key
    for key in optional_input_keys
    if not input_paths[key].exists()
]

if missing_optional_inputs:
    print("Missing optional inputs:")
    for key in missing_optional_inputs:
        print(f"- {key}: {input_paths[key]}")

source_dataframes = {}

for key, path in input_paths.items():
    if path.exists() and path.suffix.lower() == ".csv":
        source_dataframes[key] = pd.read_csv(path)
        print(f"Loaded CSV: {key} {source_dataframes[key].shape}")
    else:
        source_dataframes[key] = pd.DataFrame()

refined_comparison_df = source_dataframes["refined_comparison"]
case_selected_df = source_dataframes["case_diagnostic_selected_cases"]
case_manifest_df = source_dataframes["case_diagnostic_report_manifest"]

print("\nRequired dataframes loaded:")
print("refined_comparison_df:", refined_comparison_df.shape)
print("case_selected_df:", case_selected_df.shape)
print("case_manifest_df:", case_manifest_df.shape)

Loaded CSV: refined_comparison (200, 40)
Loaded CSV: texture_unified (750, 102)
Loaded CSV: texture_case_winners_nonzero (200, 16)
Loaded CSV: texture_winner_summary_nonzero (3, 2)
Loaded CSV: texture_disagreement_cases (200, 21)
Loaded CSV: texture_high_texture_brushwork_summary (15, 11)
Loaded CSV: brushstroke_proxy_summary_by_model (3, 6)
Loaded CSV: uncertainty_heatmap_manifest (40, 11)
Loaded CSV: uncertainty_heatmap_summary_by_case (40, 60)
Loaded CSV: uncertainty_heatmap_summary_by_mask_type (4, 10)
Loaded CSV: uncertainty_heatmap_summary_by_category (5, 10)
Loaded CSV: uncertainty_heatmap_vs_refined_performance (40, 119)
Loaded CSV: uncertainty_heatmap_selected_cases (22, 124)
Loaded CSV: case_diagnostic_selected_cases (30, 98)
Loaded CSV: case_diagnostic_report_manifest (30, 24)

Required dataframes loaded:
refined_comparison_df: (200, 40)
case_selected_df: (30, 98)
case_manifest_df: (30, 24)


In [6]:
def validate_core_dashboard_sources() -> None:
    if refined_comparison_df.empty:
        raise ValueError("Refined comparison dataframe is empty.")

    if case_selected_df.empty:
        raise ValueError("Selected case diagnostics dataframe is empty.")

    if case_manifest_df.empty:
        raise ValueError("Case diagnostic manifest dataframe is empty.")

    required_refined_columns = [
        "case_id",
        "painting_id",
        "category",
        "mask_type",
    ]

    missing_refined_columns = [
        column
        for column in required_refined_columns
        if column not in refined_comparison_df.columns
    ]

    if missing_refined_columns:
        raise ValueError(
            f"Refined comparison missing required columns: {missing_refined_columns}"
        )

    refined_case_count = refined_comparison_df["case_id"].nunique()

    if refined_case_count != 200:
        raise ValueError(
            f"Expected 200 non-zero refined comparison cases, got {refined_case_count}."
        )

    if "zero_control" in set(refined_comparison_df["mask_type"].astype(str)):
        raise ValueError("Refined comparison unexpectedly contains zero_control cases.")

    required_selected_columns = [
        "case_id",
        "selection_reasons",
        "case_diagnostic_grid_path",
        "case_report_html_path",
    ]

    missing_selected_columns = [
        column
        for column in required_selected_columns
        if column not in case_selected_df.columns
    ]

    if missing_selected_columns:
        raise ValueError(
            f"Selected case diagnostics missing required columns: {missing_selected_columns}"
        )

    missing_selected_assets = []

    for column in [
        "case_diagnostic_grid_path",
        "case_report_html_path",
    ]:
        for _, row in case_selected_df.iterrows():
            if not file_exists_from_value(row[column]):
                missing_selected_assets.append(
                    {
                        "case_id": row.get("case_id", ""),
                        "column": column,
                        "path": row[column],
                    }
                )

    if missing_selected_assets:
        display(pd.DataFrame(missing_selected_assets).head(30))
        raise FileNotFoundError("Some selected case dashboard assets are missing.")


validate_core_dashboard_sources()

print("Core dashboard source validation passed.")
print("Refined non-zero cases:", refined_comparison_df["case_id"].nunique())
print("Selected diagnostic cases:", case_selected_df["case_id"].nunique())

if "has_uncertainty_heatmap" in case_selected_df.columns:
    print("Selected cases with uncertainty heatmaps:", int(case_selected_df["has_uncertainty_heatmap"].sum()))

if "has_texture_disagreement" in case_selected_df.columns:
    print("Selected cases with texture disagreement:", int(case_selected_df["has_texture_disagreement"].sum()))

Core dashboard source validation passed.
Refined non-zero cases: 200
Selected diagnostic cases: 30
Selected cases with uncertainty heatmaps: 17
Selected cases with texture disagreement: 30


In [7]:
def safe_count_true(df: pd.DataFrame, column: str) -> int:
    if column not in df.columns:
        return 0

    return int(df[column].fillna(False).astype(bool).sum())


def count_existing_optional_file(path_key: str) -> bool:
    path = input_paths[path_key]
    return bool(path.exists())


dashboard_summary = {
    "project": {
        "title": "Trustworthy Evaluation Frameworks for AI-Assisted Painting Restoration",
        "dashboard_asset_version": "pre_feedback_final",
        "generated_by": NOTEBOOK_NAME,
        "generated_at": datetime.now().isoformat(timespec="seconds"),
    },
    "dataset": {
        "controlled_paintings": 50,
        "painting_categories": sorted(refined_comparison_df["category"].astype(str).unique().tolist()),
        "non_zero_cases": int(refined_comparison_df["case_id"].nunique()),
        "mask_types_non_zero": sorted(refined_comparison_df["mask_type"].astype(str).unique().tolist()),
        "zero_control_excluded_from_refined_comparison": True,
    },
    "models": {
        "evaluated_models": [
            "OpenCV Telea",
            "LaMa",
            "Stable Diffusion Inpainting",
        ],
        "feasibility_audited_not_fully_evaluated": [
            "SDXL Inpainting",
        ],
    },
    "available_dashboard_layers": {
        "refined_comparison": True,
        "texture_metrics": count_existing_optional_file("texture_unified"),
        "texture_winners_nonzero": count_existing_optional_file("texture_case_winners_nonzero"),
        "texture_disagreements": count_existing_optional_file("texture_disagreement_cases"),
        "uncertainty_heatmaps": count_existing_optional_file("uncertainty_heatmap_summary_by_case"),
        "case_diagnostic_reports": True,
    },
    "case_reports": {
        "selected_cases": int(case_selected_df["case_id"].nunique()),
        "selected_cases_with_uncertainty_heatmaps": safe_count_true(case_selected_df, "has_uncertainty_heatmap"),
        "selected_cases_with_texture_disagreement": safe_count_true(case_selected_df, "has_texture_disagreement"),
        "case_report_index": to_project_relative_path(input_paths["case_report_index"]),
    },
    "reports": {
        "uncertainty_heatmap_report": to_project_relative_path(input_paths["uncertainty_heatmap_report"]),
        "case_report_index": to_project_relative_path(input_paths["case_report_index"]),
    },
    "interpretation_boundaries": {
        "stable_diffusion_uncertainty": "Seed-based spatial variability, not calibrated confidence.",
        "brushstroke_proxy": "Directional texture proxy, not semantic brushstroke recognition or authentication.",
        "case_reports": "Inspection artifacts, not new metric computations.",
    },
}

write_json(
    output_paths["dashboard_summary"],
    dashboard_summary,
)

print("Saved dashboard summary:", output_paths["dashboard_summary"])
print(json.dumps(dashboard_summary, indent=2, ensure_ascii=False))

Saved dashboard summary: D:\Masters\FH\Thesis\painting-restoration-eval\outputs\dashboard\dashboard_summary.json
{
  "project": {
    "title": "Trustworthy Evaluation Frameworks for AI-Assisted Painting Restoration",
    "dashboard_asset_version": "pre_feedback_final",
    "generated_by": "34_prepare_final_dashboard_assets_cleaned",
    "generated_at": "2026-07-07T21:41:32"
  },
  "dataset": {
    "controlled_paintings": 50,
    "painting_categories": [
      "abstraction_surrealism",
      "architecture_structured",
      "high_texture_brushwork",
      "landscape_natural",
      "portrait_figure"
    ],
    "non_zero_cases": 200,
    "mask_types_non_zero": [
      "loss_large",
      "loss_small",
      "mixed_damage",
      "scratch_thin"
    ],
    "zero_control_excluded_from_refined_comparison": true
  },
  "models": {
    "evaluated_models": [
      "OpenCV Telea",
      "LaMa",
      "Stable Diffusion Inpainting"
    ],
    "feasibility_audited_not_fully_evaluated": [
      "SDX

In [8]:
def find_first_existing_column(
    df: pd.DataFrame,
    candidates: list[str],
) -> str | None:
    for column in candidates:
        if column in df.columns:
            return column

    return None


winner_column = find_first_existing_column(
    refined_comparison_df,
    [
        "overall_metric_vote",
        "refined_overall_metric_vote",
        "refined_metric_winner",
        "majority_vote_winner",
        "winner",
    ],
)

if winner_column is None:
    print("Available refined comparison columns:")
    display(pd.DataFrame({"column": refined_comparison_df.columns}))
    raise ValueError("Could not find refined winner column.")

model_winner_summary_df = (
    refined_comparison_df
    .groupby(winner_column, as_index=False)
    .agg(cases=("case_id", "count"))
    .sort_values("cases", ascending=False)
    .rename(columns={winner_column: "refined_winner"})
)

model_winner_summary_df["case_share"] = (
    model_winner_summary_df["cases"] / model_winner_summary_df["cases"].sum()
)

model_winner_summary_df.to_csv(
    output_paths["dashboard_model_winner_summary"],
    index=False,
)

print("Winner column used:", winner_column)
print("Saved:", output_paths["dashboard_model_winner_summary"])
display(model_winner_summary_df)


metric_vote_columns = [
    column
    for column in [
        "refined_opencv_metric_wins",
        "refined_lama_metric_wins",
        "refined_stable_diffusion_metric_wins",
        "opencv_telea_metric_wins",
        "lama_metric_wins",
        "stable_diffusion_inpainting_metric_wins",
    ]
    if column in refined_comparison_df.columns
]

if not metric_vote_columns:
    print("No metric vote columns found. Writing empty metric vote summary.")
    metric_vote_summary_df = pd.DataFrame(
        columns=["model", "metric_vote_column", "total_metric_votes", "mean_metric_votes"]
    )
else:
    metric_vote_rows = []

    model_name_map = {
        "refined_opencv_metric_wins": "OpenCV Telea",
        "refined_lama_metric_wins": "LaMa",
        "refined_stable_diffusion_metric_wins": "Stable Diffusion Inpainting",
        "opencv_telea_metric_wins": "OpenCV Telea",
        "lama_metric_wins": "LaMa",
        "stable_diffusion_inpainting_metric_wins": "Stable Diffusion Inpainting",
    }

    for column in metric_vote_columns:
        metric_vote_rows.append(
            {
                "model": model_name_map.get(column, column),
                "metric_vote_column": column,
                "total_metric_votes": float(refined_comparison_df[column].sum()),
                "mean_metric_votes": float(refined_comparison_df[column].mean()),
                "cases": int(refined_comparison_df[column].notna().sum()),
            }
        )

    metric_vote_summary_df = pd.DataFrame(metric_vote_rows)

metric_vote_summary_df.to_csv(
    output_paths["dashboard_metric_vote_summary"],
    index=False,
)

print("Saved:", output_paths["dashboard_metric_vote_summary"])
display(metric_vote_summary_df)

Winner column used: overall_metric_vote
Saved: D:\Masters\FH\Thesis\painting-restoration-eval\outputs\dashboard\dashboard_model_winner_summary.csv


,refined_winner,cases,case_share
0,lama,155,0.775
3,tie_lama_opencv_telea,23,0.115
1,opencv_telea,21,0.105
2,stable_diffusion_inpainting,1,0.005


Saved: D:\Masters\FH\Thesis\painting-restoration-eval\outputs\dashboard\dashboard_metric_vote_summary.csv


,model,metric_vote_column,total_metric_votes,mean_metric_votes,cases
0,OpenCV Telea,opencv_telea_metric_wins,257.0,1.285,200
1,LaMa,lama_metric_wins,924.0,4.620,200
2,Stable Diffusion Inpainting,stable_diffusion_inpainting_metric_wins,19.0,0.095,200


In [9]:
texture_summary_rows = []

texture_winner_summary_df = source_dataframes.get("texture_winner_summary_nonzero", pd.DataFrame())
brushstroke_summary_df = source_dataframes.get("brushstroke_proxy_summary_by_model", pd.DataFrame())
texture_high_texture_df = source_dataframes.get("texture_high_texture_brushwork_summary", pd.DataFrame())
texture_disagreement_df = source_dataframes.get("texture_disagreement_cases", pd.DataFrame())
texture_case_winners_df = source_dataframes.get("texture_case_winners_nonzero", pd.DataFrame())

if not texture_winner_summary_df.empty:
    texture_winner_summary_export_df = texture_winner_summary_df.copy()
    texture_winner_summary_export_df["dashboard_section"] = "texture_winner_summary_nonzero"
    texture_summary_rows.append(texture_winner_summary_export_df)

if not brushstroke_summary_df.empty:
    brushstroke_summary_export_df = brushstroke_summary_df.copy()
    brushstroke_summary_export_df["dashboard_section"] = "brushstroke_proxy_summary_by_model"
    texture_summary_rows.append(brushstroke_summary_export_df)

if not texture_high_texture_df.empty:
    high_texture_export_df = texture_high_texture_df.copy()
    high_texture_export_df["dashboard_section"] = "high_texture_brushwork_summary"
    texture_summary_rows.append(high_texture_export_df)

if texture_summary_rows:
    # Different source tables may not share columns. Concatenating with sort=False is deliberate.
    dashboard_texture_summary_df = pd.concat(
        texture_summary_rows,
        ignore_index=True,
        sort=False,
    )
else:
    dashboard_texture_summary_df = pd.DataFrame(
        columns=["dashboard_section", "note"]
    )
    dashboard_texture_summary_df.loc[0] = [
        "texture_assets_missing",
        "Texture dashboard source files were not available.",
    ]

dashboard_texture_summary_df.to_csv(
    output_paths["dashboard_texture_summary"],
    index=False,
)

print("Saved:", output_paths["dashboard_texture_summary"])
print("Texture dashboard summary shape:", dashboard_texture_summary_df.shape)
display(dashboard_texture_summary_df.head(20))


if not texture_disagreement_df.empty:
    dashboard_texture_disagreements_df = texture_disagreement_df.copy()

    if "case_id" in dashboard_texture_disagreements_df.columns:
        dashboard_texture_disagreements_df["case_id"] = dashboard_texture_disagreements_df["case_id"].astype(str)

    # Keep compact for Streamlit.
    preferred_columns = [
        column
        for column in [
            "case_id",
            "painting_id",
            "category",
            "mask_type",
            "refined_winner",
            "overall_metric_vote",
            "texture_winner",
            "texture_winner_model",
            "combined_texture_distance_winner",
            "brushstroke_proxy_distance_mean_winner",
            "texture_refined_agreement",
        ]
        if column in dashboard_texture_disagreements_df.columns
    ]

    if preferred_columns:
        dashboard_texture_disagreements_df = dashboard_texture_disagreements_df[preferred_columns].copy()
else:
    dashboard_texture_disagreements_df = pd.DataFrame(
        columns=["case_id", "note"]
    )

dashboard_texture_disagreements_df.to_csv(
    output_paths["dashboard_texture_disagreements"],
    index=False,
)

print("Saved:", output_paths["dashboard_texture_disagreements"])
print("Texture disagreement shape:", dashboard_texture_disagreements_df.shape)
display(dashboard_texture_disagreements_df.head(20))

Saved: D:\Masters\FH\Thesis\painting-restoration-eval\outputs\dashboard\dashboard_texture_summary.csv
Texture dashboard summary shape: (21, 15)


,model,nonzero_texture_wins,dashboard_section,cases,mean_brushstroke_proxy_distance,median_brushstroke_proxy_distance,mean_brushstroke_orientation_histogram_l1_distance,mean_brushstroke_orientation_coherence_difference,mask_type,mean_glcm_texture_distance,median_glcm_texture_distance,mean_gabor_texture_distance,median_gabor_texture_distance,mean_combined_texture_distance,median_combined_texture_distance
0,lama,175.0,texture_winner_summary_nonzero,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,stable_diffusion_inpainting,24.0,texture_winner_summary_nonzero,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,opencv_telea,1.0,texture_winner_summary_nonzero,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,lama,NaN,brushstroke_proxy_summary_by_model,250.0,0.006405,0.001645,0.008719,0.007470,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,opencv_telea,NaN,brushstroke_proxy_summary_by_model,250.0,0.011016,0.004184,0.014873,0.012220,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,stable_diffusion_inpainting,NaN,brushstroke_proxy_summary_by_model,250.0,0.018539,0.015303,0.029882,0.031497,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,lama,NaN,high_texture_brushwork_summary,10.0,0.024336,0.020533,NaN,NaN,loss_large,0.209704,0.139304,0.000274,0.000222,0.236821,0.182233
7,stable_diffusion_inpainting,NaN,high_texture_brushwork_summary,10.0,0.036488,0.029082,NaN,NaN,loss_large,0.195178,0.138441,0.000223,0.000169,0.243846,0.215946
8,opencv_telea,NaN,high_texture_brushwork_summary,10.0,0.038763,0.034293,NaN,NaN,loss_large,0.311879,0.271550,0.000404,0.000404,0.357130,0.344861
9,lama,NaN,high_texture_brushwork_summary,10.0,0.001665,0.001299,NaN,NaN,loss_small,0.005619,0.003014,0.000008,0.000006,0.009235,0.006918


Saved: D:\Masters\FH\Thesis\painting-restoration-eval\outputs\dashboard\dashboard_texture_disagreements.csv
Texture disagreement shape: (200, 6)


,case_id,painting_id,category,mask_type,overall_metric_vote,texture_winner_model
0,p043_loss_large,p043,high_texture_brushwork,loss_large,lama,stable_diffusion_inpainting
1,p033_loss_large,p033,abstraction_surrealism,loss_large,lama,stable_diffusion_inpainting
2,p040_loss_large,p040,abstraction_surrealism,loss_large,opencv_telea,stable_diffusion_inpainting
3,p039_loss_large,p039,abstraction_surrealism,loss_large,lama,stable_diffusion_inpainting
4,p046_loss_large,p046,high_texture_brushwork,loss_large,lama,stable_diffusion_inpainting
5,p005_loss_large,p005,portrait_figure,loss_large,lama,stable_diffusion_inpainting
6,p030_loss_large,p030,architecture_structured,loss_large,lama,stable_diffusion_inpainting
7,p045_loss_large,p045,high_texture_brushwork,loss_large,lama,stable_diffusion_inpainting
8,p035_loss_large,p035,abstraction_surrealism,loss_large,lama,stable_diffusion_inpainting
9,p009_loss_large,p009,portrait_figure,loss_large,lama,stable_diffusion_inpainting


In [10]:
uncertainty_by_case_df = source_dataframes.get("uncertainty_heatmap_summary_by_case", pd.DataFrame())
uncertainty_by_mask_df = source_dataframes.get("uncertainty_heatmap_summary_by_mask_type", pd.DataFrame())
uncertainty_by_category_df = source_dataframes.get("uncertainty_heatmap_summary_by_category", pd.DataFrame())
uncertainty_selected_df = source_dataframes.get("uncertainty_heatmap_selected_cases", pd.DataFrame())

uncertainty_summary_frames = []

if not uncertainty_by_mask_df.empty:
    mask_export_df = uncertainty_by_mask_df.copy()
    mask_export_df["dashboard_section"] = "uncertainty_by_mask_type"
    uncertainty_summary_frames.append(mask_export_df)

if not uncertainty_by_category_df.empty:
    category_export_df = uncertainty_by_category_df.copy()
    category_export_df["dashboard_section"] = "uncertainty_by_category"
    uncertainty_summary_frames.append(category_export_df)

if uncertainty_summary_frames:
    dashboard_uncertainty_summary_df = pd.concat(
        uncertainty_summary_frames,
        ignore_index=True,
        sort=False,
    )
else:
    dashboard_uncertainty_summary_df = pd.DataFrame(
        columns=["dashboard_section", "note"]
    )
    dashboard_uncertainty_summary_df.loc[0] = [
        "uncertainty_assets_missing",
        "Uncertainty heatmap summary source files were not available.",
    ]

dashboard_uncertainty_summary_df.to_csv(
    output_paths["dashboard_uncertainty_summary"],
    index=False,
)

print("Saved:", output_paths["dashboard_uncertainty_summary"])
print("Uncertainty summary shape:", dashboard_uncertainty_summary_df.shape)
display(dashboard_uncertainty_summary_df.head(20))


if not uncertainty_selected_df.empty:
    dashboard_uncertainty_selected_cases_df = uncertainty_selected_df.copy()

    if "case_id" in dashboard_uncertainty_selected_cases_df.columns:
        dashboard_uncertainty_selected_cases_df["case_id"] = dashboard_uncertainty_selected_cases_df["case_id"].astype(str)

    preferred_columns = [
        column
        for column in [
            "case_id",
            "painting_id",
            "category",
            "mask_type",
            "selection_reasons",
            "selection_count",
            "masked_mean_uncertainty",
            "boundary_mean_uncertainty",
            "outside_mask_mean_uncertainty",
            "heatmap_png_path",
            "overlay_png_path",
            "selected_detail_figure_path",
        ]
        if column in dashboard_uncertainty_selected_cases_df.columns
    ]

    if preferred_columns:
        dashboard_uncertainty_selected_cases_df = dashboard_uncertainty_selected_cases_df[
            preferred_columns
        ].copy()
else:
    dashboard_uncertainty_selected_cases_df = pd.DataFrame(
        columns=["case_id", "note"]
    )

dashboard_uncertainty_selected_cases_df.to_csv(
    output_paths["dashboard_uncertainty_selected_cases"],
    index=False,
)

print("Saved:", output_paths["dashboard_uncertainty_selected_cases"])
print("Uncertainty selected cases shape:", dashboard_uncertainty_selected_cases_df.shape)
display(dashboard_uncertainty_selected_cases_df.head(20))

Saved: D:\Masters\FH\Thesis\painting-restoration-eval\outputs\dashboard\dashboard_uncertainty_summary.csv
Uncertainty summary shape: (9, 12)


,mask_type,cases,mean_global_uncertainty,mean_masked_uncertainty,median_masked_uncertainty,mean_bbox_uncertainty,mean_boundary_uncertainty,mean_outside_mask_uncertainty,mean_masked_vs_global_ratio,mean_boundary_vs_global_ratio,dashboard_section,category
0,loss_large,10,4.750954,24.622568,22.084580,14.702201,6.614203,2.546994,5.362492,1.419112,uncertainty_by_mask_type,NaN
1,mixed_damage,10,4.050491,17.688853,17.553786,4.718454,6.024136,2.896445,4.499372,1.461297,uncertainty_by_mask_type,NaN
2,loss_small,10,2.985344,16.047955,15.562322,3.971199,6.301782,2.494192,5.500481,2.095952,uncertainty_by_mask_type,NaN
3,scratch_thin,10,2.860343,12.074715,11.957541,3.267405,5.300038,2.692350,4.432006,1.887867,uncertainty_by_mask_type,NaN
4,NaN,8,3.711135,20.055819,21.057585,6.663922,5.623687,2.446155,5.682910,1.594331,uncertainty_by_category,abstraction_surrealism
5,NaN,8,3.005084,19.835183,17.513558,6.691588,4.645026,1.845033,6.442190,1.656302,uncertainty_by_category,landscape_natural
6,NaN,8,4.619064,18.022746,15.380100,7.594927,7.170386,3.537830,3.835811,1.632620,uncertainty_by_category,portrait_figure
7,NaN,8,3.856592,15.738840,16.219893,6.582487,7.371399,3.011359,4.160744,1.901402,uncertainty_by_category,high_texture_brushwork
8,NaN,8,3.117041,14.390026,13.334217,5.791148,5.489702,2.447099,4.621284,1.795631,uncertainty_by_category,architecture_structured


Saved: D:\Masters\FH\Thesis\painting-restoration-eval\outputs\dashboard\dashboard_uncertainty_selected_cases.csv
Uncertainty selected cases shape: (22, 12)


,case_id,painting_id,category,mask_type,selection_reasons,selection_count,masked_mean_uncertainty,boundary_mean_uncertainty,outside_mask_mean_uncertainty,heatmap_png_path,overlay_png_path,selected_detail_figure_path
0,p011_loss_large,p011,landscape_natural,loss_large,category_representative_high_uncertainty__land...,4,38.245800,4.813465,1.559997,outputs/figures/uncertainty_heatmaps/heatmap_i...,outputs/figures/uncertainty_heatmaps/overlay_i...,outputs/figures/uncertainty_heatmaps/selected_...
1,p009_loss_large,p009,portrait_figure,loss_large,category_representative_high_uncertainty__port...,4,37.621746,8.283583,3.798770,outputs/figures/uncertainty_heatmaps/heatmap_i...,outputs/figures/uncertainty_heatmaps/overlay_i...,outputs/figures/uncertainty_heatmaps/selected_...
2,p031_loss_large,p031,abstraction_surrealism,loss_large,category_representative_high_uncertainty__abst...,3,27.727493,7.723886,3.094143,outputs/figures/uncertainty_heatmaps/heatmap_i...,outputs/figures/uncertainty_heatmaps/overlay_i...,outputs/figures/uncertainty_heatmaps/selected_...
3,p037_loss_large,p037,abstraction_surrealism,loss_large,highest_masked_uncertainty; uncertainty_perfor...,2,25.150114,3.572868,1.621859,outputs/figures/uncertainty_heatmaps/heatmap_i...,outputs/figures/uncertainty_heatmaps/overlay_i...,outputs/figures/uncertainty_heatmaps/selected_...
4,p011_mixed_damage,p011,landscape_natural,mixed_damage,highest_masked_uncertainty; mask_type_represen...,2,24.741758,4.385314,1.783683,outputs/figures/uncertainty_heatmaps/heatmap_i...,outputs/figures/uncertainty_heatmaps/overlay_i...,outputs/figures/uncertainty_heatmaps/selected_...
5,p037_loss_small,p037,abstraction_surrealism,loss_small,mask_type_representative_high_uncertainty__los...,2,22.331778,5.407199,1.794383,outputs/figures/uncertainty_heatmaps/heatmap_i...,outputs/figures/uncertainty_heatmaps/overlay_i...,outputs/figures/uncertainty_heatmaps/selected_...
6,p041_mixed_damage,p041,high_texture_brushwork,mixed_damage,highest_boundary_uncertainty; highest_outside_...,2,18.264513,10.275064,3.833238,outputs/figures/uncertainty_heatmaps/heatmap_i...,outputs/figures/uncertainty_heatmaps/overlay_i...,outputs/figures/uncertainty_heatmaps/selected_...
7,p041_scratch_thin,p041,high_texture_brushwork,scratch_thin,highest_outside_mask_uncertainty; lowest_maske...,2,10.916192,7.015200,3.933914,outputs/figures/uncertainty_heatmaps/heatmap_i...,outputs/figures/uncertainty_heatmaps/overlay_i...,outputs/figures/uncertainty_heatmaps/selected_...
8,p037_mixed_damage,p037,abstraction_surrealism,mixed_damage,uncertainty_performance_quadrant__middle_region,1,22.276716,4.110384,1.942242,outputs/figures/uncertainty_heatmaps/heatmap_i...,outputs/figures/uncertainty_heatmaps/overlay_i...,outputs/figures/uncertainty_heatmaps/selected_...
9,p043_loss_large,p043,high_texture_brushwork,loss_large,category_representative_high_uncertainty__high...,1,21.625156,6.009873,2.034851,outputs/figures/uncertainty_heatmaps/heatmap_i...,outputs/figures/uncertainty_heatmaps/overlay_i...,outputs/figures/uncertainty_heatmaps/selected_...


In [11]:
def normalize_dashboard_path_columns(
    df: pd.DataFrame,
    path_columns: list[str],
) -> pd.DataFrame:
    normalized_df = df.copy()

    for column in path_columns:
        if column in normalized_df.columns:
            normalized_df[column] = normalized_df[column].apply(to_project_relative_path)
            normalized_df[f"{column}_exists"] = normalized_df[column].apply(file_exists_from_value)

    return normalized_df


case_report_manifest_columns = [
    column
    for column in [
        "case_id",
        "painting_id",
        "mask_id",
        "category",
        "mask_type",
        "selection_reasons",
        "selection_count",
        "has_uncertainty_heatmap",
        "has_texture_disagreement",
        "case_diagnostic_grid_path",
        "case_report_html_path",
        "clean_image_path_relative",
        "damaged_image_path_relative",
        "mask_image_path_relative",
        "opencv_image_path_relative",
        "lama_image_path_relative",
        "stable_diffusion_image_path_relative",
        "uncertainty_heatmap_png_path",
        "uncertainty_overlay_png_path",
        "uncertainty_masked_mean_uncertainty",
        "uncertainty_boundary_mean_uncertainty",
        "refined_stable_diffusion_metric_wins",
        "refined_opencv_metric_wins",
        "refined_lama_metric_wins",
    ]
    if column in case_manifest_df.columns
]

dashboard_case_report_manifest_df = case_manifest_df[case_report_manifest_columns].copy()

dashboard_case_report_manifest_df = normalize_dashboard_path_columns(
    dashboard_case_report_manifest_df,
    [
        "case_diagnostic_grid_path",
        "case_report_html_path",
        "clean_image_path_relative",
        "damaged_image_path_relative",
        "mask_image_path_relative",
        "opencv_image_path_relative",
        "lama_image_path_relative",
        "stable_diffusion_image_path_relative",
        "uncertainty_heatmap_png_path",
        "uncertainty_overlay_png_path",
    ],
)

dashboard_case_report_manifest_df.to_csv(
    output_paths["dashboard_case_report_manifest"],
    index=False,
)

print("Saved:", output_paths["dashboard_case_report_manifest"])
print("Dashboard case report manifest shape:", dashboard_case_report_manifest_df.shape)
display(dashboard_case_report_manifest_df.head(20))


selected_case_columns = [
    column
    for column in [
        "case_id",
        "category",
        "mask_type",
        "selection_reasons",
        "selection_count",
        "has_uncertainty_heatmap",
        "has_texture_disagreement",
        "case_diagnostic_grid_path",
        "case_report_html_path",
        "uncertainty_masked_mean_uncertainty",
        "uncertainty_boundary_mean_uncertainty",
        "uncertainty_outside_mask_mean_uncertainty",
        "refined_stable_diffusion_metric_wins",
        "refined_opencv_metric_wins",
        "refined_lama_metric_wins",
    ]
    if column in case_selected_df.columns
]

dashboard_selected_cases_df = case_selected_df[selected_case_columns].copy()

dashboard_selected_cases_df = normalize_dashboard_path_columns(
    dashboard_selected_cases_df,
    [
        "case_diagnostic_grid_path",
        "case_report_html_path",
    ],
)

dashboard_selected_cases_df.to_csv(
    output_paths["dashboard_selected_cases"],
    index=False,
)

print("Saved:", output_paths["dashboard_selected_cases"])
print("Dashboard selected cases shape:", dashboard_selected_cases_df.shape)
display(dashboard_selected_cases_df.head(20))

Saved: D:\Masters\FH\Thesis\painting-restoration-eval\outputs\dashboard\dashboard_case_report_manifest.csv
Dashboard case report manifest shape: (30, 31)


,case_id,painting_id,mask_id,category,mask_type,selection_reasons,selection_count,has_uncertainty_heatmap,has_texture_disagreement,case_diagnostic_grid_path,case_report_html_path,clean_image_path_relative,damaged_image_path_relative,mask_image_path_relative,opencv_image_path_relative,lama_image_path_relative,stable_diffusion_image_path_relative,uncertainty_heatmap_png_path,uncertainty_overlay_png_path,uncertainty_masked_mean_uncertainty,uncertainty_boundary_mean_uncertainty,case_diagnostic_grid_path_exists,case_report_html_path_exists,clean_image_path_relative_exists,damaged_image_path_relative_exists,mask_image_path_relative_exists,opencv_image_path_relative_exists,lama_image_path_relative_exists,stable_diffusion_image_path_relative_exists,uncertainty_heatmap_png_path_exists,uncertainty_overlay_png_path_exists
0,p009_loss_large,p009,p009_loss_large,portrait_figure,loss_large,category_representative__portrait_figure; high...,3,True,True,outputs/figures/case_diagnostics/selected_case...,outputs/reports/case_diagnostics/selected_case...,data/processed/clean/p009_clean.png,data/processed/masked/p009_loss_large_damaged.png,data/processed/masks/p009_loss_large_mask.png,data/processed/restored/opencv_telea/p009_loss...,data/processed/restored/lama/p009_loss_large_r...,data/processed/restored/stable_diffusion_inpai...,outputs/figures/uncertainty_heatmaps/heatmap_i...,outputs/figures/uncertainty_heatmaps/overlay_i...,37.621746,8.283583,True,True,True,True,True,True,True,True,True,True
1,p011_loss_large,p011,p011_loss_large,landscape_natural,loss_large,category_representative__landscape_natural; hi...,3,True,True,outputs/figures/case_diagnostics/selected_case...,outputs/reports/case_diagnostics/selected_case...,data/processed/clean/p011_clean.png,data/processed/masked/p011_loss_large_damaged.png,data/processed/masks/p011_loss_large_mask.png,data/processed/restored/opencv_telea/p011_loss...,data/processed/restored/lama/p011_loss_large_r...,data/processed/restored/stable_diffusion_inpai...,outputs/figures/uncertainty_heatmaps/heatmap_i...,outputs/figures/uncertainty_heatmaps/overlay_i...,38.245800,4.813465,True,True,True,True,True,True,True,True,True,True
2,p001_loss_large,p001,p001_loss_large,portrait_figure,loss_large,lama_strong_refined_metric_wins; stable_diffus...,3,False,True,outputs/figures/case_diagnostics/selected_case...,outputs/reports/case_diagnostics/selected_case...,data/processed/clean/p001_clean.png,data/processed/masked/p001_loss_large_damaged.png,data/processed/masks/p001_loss_large_mask.png,data/processed/restored/opencv_telea/p001_loss...,data/processed/restored/lama/p001_loss_large_r...,data/processed/restored/stable_diffusion_inpai...,None,None,NaN,NaN,True,True,True,True,True,True,True,True,False,False
3,p011_mixed_damage,p011,p011_mixed_damage,landscape_natural,mixed_damage,highest_stable_diffusion_masked_uncertainty; m...,2,True,True,outputs/figures/case_diagnostics/selected_case...,outputs/reports/case_diagnostics/selected_case...,data/processed/clean/p011_clean.png,data/processed/masked/p011_mixed_damage_damage...,data/processed/masks/p011_mixed_damage_mask.png,data/processed/restored/opencv_telea/p011_mixe...,data/processed/restored/lama/p011_mixed_damage...,data/processed/restored/stable_diffusion_inpai...,outputs/figures/uncertainty_heatmaps/heatmap_i...,outputs/figures/uncertainty_heatmaps/overlay_i...,24.741758,4.385314,True,True,True,True,True,True,True,True,True,True
4,p031_loss_large,p031,p031_loss_large,abstraction_surrealism,loss_large,category_representative__abstraction_surrealis...,2,True,True,outputs/figures/case_diagnostics/selected_case...,outputs/reports/case_diagnostics/selected_case...,data/processed/clean/p031_clean.png,data/processed/masked/p031_loss_large_damaged.png,data/processed/masks/p031_loss_large_mask.png,data/processed/restored/opencv_telea/p031_loss...,data/processed/restored/lama/p031_loss_large_r...,data/processed/restored/stable_diffusion_inpai...,outputs/figures/uncertainty_he

Saved: D:\Masters\FH\Thesis\painting-restoration-eval\outputs\dashboard\dashboard_selected_cases.csv
Dashboard selected cases shape: (30, 14)


,case_id,category,mask_type,selection_reasons,selection_count,has_uncertainty_heatmap,has_texture_disagreement,case_diagnostic_grid_path,case_report_html_path,uncertainty_masked_mean_uncertainty,uncertainty_boundary_mean_uncertainty,uncertainty_outside_mask_mean_uncertainty,case_diagnostic_grid_path_exists,case_report_html_path_exists
0,p009_loss_large,portrait_figure,loss_large,category_representative__portrait_figure; high...,3,True,True,outputs/figures/case_diagnostics/selected_case...,outputs/reports/case_diagnostics/selected_case...,37.621746,8.283583,3.798770,True,True
1,p011_loss_large,landscape_natural,loss_large,category_representative__landscape_natural; hi...,3,True,True,outputs/figures/case_diagnostics/selected_case...,outputs/reports/case_diagnostics/selected_case...,38.245800,4.813465,1.559997,True,True
2,p001_loss_large,portrait_figure,loss_large,lama_strong_refined_metric_wins; stable_diffus...,3,False,True,outputs/figures/case_diagnostics/selected_case...,outputs/reports/case_diagnostics/selected_case...,NaN,NaN,NaN,True,True
3,p011_mixed_damage,landscape_natural,mixed_damage,highest_stable_diffusion_masked_uncertainty; m...,2,True,True,outputs/figures/case_diagnostics/selected_case...,outputs/reports/case_diagnostics/selected_case...,24.741758,4.385314,1.783683,True,True
4,p031_loss_large,abstraction_surrealism,loss_large,category_representative__abstraction_surrealis...,2,True,True,outputs/figures/case_diagnostics/selected_case...,outputs/reports/case_diagnostics/selected_case...,27.727493,7.723886,3.094143,True,True
5,p041_loss_large,high_texture_brushwork,loss_large,high_texture_brushwork_representative; highest...,2,True,True,outputs/figures/case_diagnostics/selected_case...,outputs/reports/case_diagnostics/selected_case...,16.199604,10.067053,3.367184,True,True
6,p041_loss_small,high_texture_brushwork,loss_small,high_texture_brushwork_representative; highest...,2,True,True,outputs/figures/case_diagnostics/selected_case...,outputs/reports/case_diagnostics/selected_case...,13.953583,9.221172,3.316785,True,True
7,p041_mixed_damage,high_texture_brushwork,mixed_damage,high_texture_brushwork_representative; highest...,2,True,True,outputs/figures/case_diagnostics/selected_case...,outputs/reports/case_diagnostics/selected_case...,18.264513,10.275064,3.833238,True,True
8,p043_loss_large,high_texture_brushwork,loss_large,category_representative__high_texture_brushwor...,2,True,True,outputs/figures/case_diagnostics/selected_case...,outputs/reports/case_diagnostics/selected_case...,21.625156,6.009873,2.034851,True,True
9,p009_mixed_damage,portrait_figure,mixed_damage,highest_stable_diffusion_boundary_uncertainty,1,True,True,outputs/figures/case_diagnostics/selected_case...,outputs/reports/case_diagnostics/selected_case...,21.450689,8.090476,3.770992,True,True


In [12]:
figure_manifest_rows = []

figure_search_roots = [
    figures_dir / "uncertainty_heatmaps",
    figures_dir / "case_diagnostics",
]

for root in figure_search_roots:
    if not root.exists():
        continue

    for path in sorted(root.rglob("*.png")):
        figure_manifest_rows.append(
            {
                "asset_type": "figure",
                "asset_group": path.parent.name,
                "asset_name": path.name,
                "asset_path": to_project_relative_path(path),
                "exists": path.exists(),
                "file_size_kb": round(path.stat().st_size / 1024, 2),
            }
        )

report_manifest_rows = []

for report_key in [
    "uncertainty_heatmap_report",
    "case_report_index",
]:
    report_path = input_paths[report_key]

    report_manifest_rows.append(
        {
            "asset_type": "report",
            "asset_group": report_key,
            "asset_name": report_path.name,
            "asset_path": to_project_relative_path(report_path),
            "exists": report_path.exists(),
            "file_size_kb": round(report_path.stat().st_size / 1024, 2) if report_path.exists() else np.nan,
        }
    )

dashboard_figure_manifest_df = pd.DataFrame(
    figure_manifest_rows + report_manifest_rows
)

dashboard_figure_manifest_df.to_csv(
    output_paths["dashboard_figure_manifest"],
    index=False,
)

print("Saved:", output_paths["dashboard_figure_manifest"])
print("Figure/report manifest shape:", dashboard_figure_manifest_df.shape)

display(dashboard_figure_manifest_df.head(30))

Saved: D:\Masters\FH\Thesis\painting-restoration-eval\outputs\dashboard\dashboard_figure_manifest.csv
Figure/report manifest shape: (180, 6)


,asset_type,asset_group,asset_name,asset_path,exists,file_size_kb
0,figure,all_cases,p009_loss_large_uncertainty_compact.png,outputs/figures/uncertainty_heatmaps/all_cases...,True,1252.68
1,figure,all_cases,p009_loss_small_uncertainty_compact.png,outputs/figures/uncertainty_heatmaps/all_cases...,True,1271.84
2,figure,all_cases,p009_mixed_damage_uncertainty_compact.png,outputs/figures/uncertainty_heatmaps/all_cases...,True,1306.35
3,figure,all_cases,p009_scratch_thin_uncertainty_compact.png,outputs/figures/uncertainty_heatmaps/all_cases...,True,1322.91
4,figure,all_cases,p010_loss_large_uncertainty_compact.png,outputs/figures/uncertainty_heatmaps/all_cases...,True,1288.18
5,figure,all_cases,p010_loss_small_uncertainty_compact.png,outputs/figures/uncertainty_heatmaps/all_cases...,True,1272.40
6,figure,all_cases,p010_mixed_damage_uncertainty_compact.png,outputs/figures/uncertainty_heatmaps/all_cases...,True,1364.33
7,figure,all_cases,p010_scratch_thin_uncertainty_compact.png,outputs/figures/uncertainty_heatmaps/all_cases...,True,1323.62
8,figure,all_cases,p011_loss_large_uncertainty_compact.png,outputs/figures/uncertainty_heatmaps/all_cases...,True,862.50
9,figure,all_cases,p011_loss_small_uncertainty_compact.png,outputs/figures/uncertainty_heatmaps/all_cases...,True,867.85


In [15]:
dashboard_asset_files = {}

for key, path in output_paths.items():
    # The manifest is being written by this cell, so it cannot be checked
    # before it exists. Mark it as expected and refresh after writing.
    if key == "dashboard_asset_manifest":
        dashboard_asset_files[key] = {
            "path": to_project_relative_path(path),
            "exists": True,
            "file_size_kb": None,
        }
        continue

    dashboard_asset_files[key] = {
        "path": to_project_relative_path(path),
        "exists": path.exists(),
        "file_size_kb": round(path.stat().st_size / 1024, 2) if path.exists() else None,
    }

dashboard_asset_manifest = {
    "project": dashboard_summary["project"],
    "asset_directory": to_project_relative_path(dashboard_dir),
    "assets": dashboard_asset_files,
    "source_inputs": {
        key: {
            "path": to_project_relative_path(path),
            "exists": path.exists(),
        }
        for key, path in input_paths.items()
    },
    "dashboard_sections": [
        "overview",
        "model_comparison",
        "texture_brushstroke_proxy",
        "stable_diffusion_uncertainty",
        "case_reports",
        "limitations",
    ],
}

write_json(
    output_paths["dashboard_asset_manifest"],
    dashboard_asset_manifest,
)

# Refresh the self-entry now that the file exists.
dashboard_asset_manifest["assets"]["dashboard_asset_manifest"] = {
    "path": to_project_relative_path(output_paths["dashboard_asset_manifest"]),
    "exists": output_paths["dashboard_asset_manifest"].exists(),
    "file_size_kb": round(output_paths["dashboard_asset_manifest"].stat().st_size / 1024, 2),
}

write_json(
    output_paths["dashboard_asset_manifest"],
    dashboard_asset_manifest,
)

print("Saved:", output_paths["dashboard_asset_manifest"])
print(json.dumps(dashboard_asset_manifest, indent=2, ensure_ascii=False))

Saved: D:\Masters\FH\Thesis\painting-restoration-eval\outputs\dashboard\dashboard_asset_manifest.json
{
  "project": {
    "title": "Trustworthy Evaluation Frameworks for AI-Assisted Painting Restoration",
    "dashboard_asset_version": "pre_feedback_final",
    "generated_by": "34_prepare_final_dashboard_assets_cleaned",
    "generated_at": "2026-07-07T21:41:32"
  },
  "asset_directory": "outputs/dashboard",
  "assets": {
    "dashboard_summary": {
      "path": "outputs/dashboard/dashboard_summary.json",
      "exists": true,
      "file_size_kb": 1.93
    },
    "dashboard_model_winner_summary": {
      "path": "outputs/dashboard/dashboard_model_winner_summary.csv",
      "exists": true,
      "file_size_kb": 0.14
    },
    "dashboard_metric_vote_summary": {
      "path": "outputs/dashboard/dashboard_metric_vote_summary.csv",
      "exists": true,
      "file_size_kb": 0.24
    },
    "dashboard_texture_summary": {
      "path": "outputs/dashboard/dashboard_texture_summary.csv",
  

In [16]:
required_dashboard_outputs = [
    output_paths["dashboard_summary"],
    output_paths["dashboard_model_winner_summary"],
    output_paths["dashboard_metric_vote_summary"],
    output_paths["dashboard_texture_summary"],
    output_paths["dashboard_texture_disagreements"],
    output_paths["dashboard_uncertainty_summary"],
    output_paths["dashboard_uncertainty_selected_cases"],
    output_paths["dashboard_case_report_manifest"],
    output_paths["dashboard_selected_cases"],
    output_paths["dashboard_figure_manifest"],
    output_paths["dashboard_asset_manifest"],
]

missing_dashboard_outputs = [
    path
    for path in required_dashboard_outputs
    if not path.exists()
]

if missing_dashboard_outputs:
    for path in missing_dashboard_outputs:
        print("Missing:", path)

    raise FileNotFoundError("Some required dashboard outputs are missing.")

if refined_comparison_df["case_id"].nunique() != 200:
    raise ValueError(
        f"Expected 200 non-zero refined cases, got {refined_comparison_df['case_id'].nunique()}."
    )

if dashboard_selected_cases_df.empty:
    raise ValueError("Dashboard selected cases output is empty.")

if dashboard_case_report_manifest_df.empty:
    raise ValueError("Dashboard case report manifest output is empty.")

for column in [
    "case_diagnostic_grid_path_exists",
    "case_report_html_path_exists",
]:
    if column in dashboard_selected_cases_df.columns:
        if not dashboard_selected_cases_df[column].all():
            display(
                dashboard_selected_cases_df[
                    ~dashboard_selected_cases_df[column]
                ][["case_id", column]]
            )
            raise FileNotFoundError(f"Some selected case assets are missing for {column}.")

summary_payload = json.loads(output_paths["dashboard_summary"].read_text(encoding="utf-8"))
asset_manifest_payload = json.loads(output_paths["dashboard_asset_manifest"].read_text(encoding="utf-8"))

if summary_payload["dataset"]["non_zero_cases"] != 200:
    raise ValueError("Dashboard summary JSON has incorrect non-zero case count.")

missing_manifest_assets = [
    key
    for key, metadata in asset_manifest_payload["assets"].items()
    if not metadata["exists"]
]

if missing_manifest_assets:
    raise FileNotFoundError(f"Dashboard asset manifest has missing assets: {missing_manifest_assets}")

# Cross-check manifest self-entry. Because apparently existence needs paperwork.
manifest_entry = asset_manifest_payload["assets"].get("dashboard_asset_manifest", {})

if not manifest_entry.get("exists", False):
    raise ValueError("dashboard_asset_manifest self-entry is not marked as existing.")

print("Notebook 34 final validation passed.")
print("Dashboard directory:", dashboard_dir)
print("Dashboard asset count:", len(asset_manifest_payload["assets"]))
print("Selected dashboard cases:", len(dashboard_selected_cases_df))
print("Figure/report assets:", len(dashboard_figure_manifest_df))

print("\nDashboard outputs:")
for path in required_dashboard_outputs:
    print("-", path, "|", round(path.stat().st_size / 1024, 2), "KB")

Notebook 34 final validation passed.
Dashboard directory: D:\Masters\FH\Thesis\painting-restoration-eval\outputs\dashboard
Dashboard asset count: 11
Selected dashboard cases: 30
Figure/report assets: 180

Dashboard outputs:
- D:\Masters\FH\Thesis\painting-restoration-eval\outputs\dashboard\dashboard_summary.json | 1.93 KB
- D:\Masters\FH\Thesis\painting-restoration-eval\outputs\dashboard\dashboard_model_winner_summary.csv | 0.14 KB
- D:\Masters\FH\Thesis\painting-restoration-eval\outputs\dashboard\dashboard_metric_vote_summary.csv | 0.24 KB
- D:\Masters\FH\Thesis\painting-restoration-eval\outputs\dashboard\dashboard_texture_summary.csv | 3.9 KB
- D:\Masters\FH\Thesis\painting-restoration-eval\outputs\dashboard\dashboard_texture_disagreements.csv | 14.05 KB
- D:\Masters\FH\Thesis\painting-restoration-eval\outputs\dashboard\dashboard_uncertainty_summary.csv | 1.93 KB
- D:\Masters\FH\Thesis\painting-restoration-eval\outputs\dashboard\dashboard_uncertainty_selected_cases.csv | 10.38 KB
- D

## Notebook 34 summary

This notebook prepared dashboard-ready assets for the final pre-feedback restoration evaluation framework.

The dashboard assets combine:

- refined model comparison results,
- model winner and metric-vote summaries,
- texture and brushstroke-proxy diagnostics,
- Stable Diffusion uncertainty heatmap summaries,
- selected case diagnostic reports.

The notebook does not rerun restoration models and does not recompute core metrics.

Main outputs:

- `outputs/dashboard/dashboard_summary.json`
- `outputs/dashboard/dashboard_model_winner_summary.csv`
- `outputs/dashboard/dashboard_metric_vote_summary.csv`
- `outputs/dashboard/dashboard_texture_summary.csv`
- `outputs/dashboard/dashboard_texture_disagreements.csv`
- `outputs/dashboard/dashboard_uncertainty_summary.csv`
- `outputs/dashboard/dashboard_uncertainty_selected_cases.csv`
- `outputs/dashboard/dashboard_case_report_manifest.csv`
- `outputs/dashboard/dashboard_selected_cases.csv`
- `outputs/dashboard/dashboard_figure_manifest.csv`
- `outputs/dashboard/dashboard_asset_manifest.json`

These files are intended to be consumed by `streamlit_app.py`.

The Streamlit dashboard should present:

- project overview,
- refined model comparison,
- texture and brushstroke-proxy diagnostics,
- Stable Diffusion uncertainty heatmap analysis,
- selected case diagnostic reports,
- interpretation boundaries and limitations.

The dashboard should treat uncertainty heatmaps as seed-based spatial variability diagnostics, not calibrated confidence estimates. Brushstroke-proxy metrics should be described as directional texture proxies, not semantic brushstroke recognition.